# Housing Affordability and Tenure Analysis

This notebook uses the prepared `master.csv` file to estimate survey-weighted housing affordability indicators by year, tenure group, income position, and urban geography.


## Data Sources

The household survey files used here are the cleaned Iranian Household Budget Survey files for 1394-1403 published by D-Learn: <https://d-learn.ir/iran-hbs/>. The page describes the files as cleaned and harmonized CSV/XLSX versions of the Statistical Center of Iran Household Expenditure and Income Survey.

CPI information is taken from the World Bank indicator `FP.CPI.TOTL.ZG` for Iran: <https://data.worldbank.org/indicator/FP.CPI.TOTL.ZG?end=2024&locations=IR&start=1960>. The local workbook `CPI.xlsx` is used as the project copy for the CPI merge.

The repository should treat the files in this folder as derived working data. If raw or cleaned survey files are not redistributed, the links above identify where the source data can be obtained.


## Scope and Limitations

The data are treated as repeated annual cross-sections, not as a household panel. Tenure groups are assigned from text patterns in the `Tenure` field, which is transparent and reproducible but still depends on the source naming convention. The adjusted housing-cost measure removes imputed rent for owner and mortgage households; this choice is appropriate for the burden measures used here, but alternative definitions may be useful for other research questions.

The CPI adjustment is used to compare values over time. Because the analysis combines local CPI data with household survey data, the exact CPI source and base year should be reported with any table or figure derived from these notebooks.


In [ ]:
# Shared definitions used across the analysis.
# Source-language strings appear only where they are needed to match raw data values.

from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR_CANDIDATES = [
    Path("../data/cleaned"),
    Path("data/cleaned"),
]
DATA_DIR = next(
    (candidate.resolve() for candidate in DATA_DIR_CANDIDATES if candidate.exists()),
    DATA_DIR_CANDIDATES[0].resolve(),
)
HOUSING_COL = "Housing, Water, Electricity, Gas and Other Fuels"
TEHRAN_LABELS = {"tehran", "تهران"}
OWNER_PATTERN = "own"
RENTER_PATTERN = "rent|mortg"


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with stripped string column names."""
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()
    return out


def coerce_numeric(df: pd.DataFrame, columns: list[str], fill_missing: float | None = None) -> pd.DataFrame:
    """Coerce selected columns to numeric, optionally creating missing columns."""
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            if fill_missing is None:
                raise KeyError(f"Missing required column: {col}")
            out[col] = fill_missing
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    """Weighted mean that ignores missing values and non-positive weights."""
    mask = values.notna() & weights.notna() & (weights > 0)
    if not mask.any():
        return np.nan
    return np.average(values[mask], weights=weights[mask])


def add_urban_rural_label(df: pd.DataFrame, source_col: str = "Urban_Rural", target_col: str = "UR") -> pd.DataFrame:
    out = df.copy()
    ur = out[source_col].astype(str).str.strip().str.lower()
    out[target_col] = pd.NA
    out.loc[ur.str.startswith("u"), target_col] = "Urban"
    out.loc[ur.str.startswith("r"), target_col] = "Rural"
    return out[out[target_col].isin(["Urban", "Rural"])].copy()


def add_tenure_group(df: pd.DataFrame, source_col: str = "Tenure", target_col: str = "tenure_group") -> pd.DataFrame:
    out = df.copy()
    tenure = out[source_col].astype(str).str.strip().str.lower()
    out[target_col] = pd.NA
    out.loc[tenure.str.contains(RENTER_PATTERN, na=False), target_col] = "Renter"
    out.loc[tenure.str.contains(OWNER_PATTERN, na=False), target_col] = "Owner"
    return out[out[target_col].isin(["Owner", "Renter"])].copy()


def add_urban_tehran_group(df: pd.DataFrame, ur_col: str = "UR", target_col: str = "UR_EXT") -> pd.DataFrame:
    out = df.copy()
    province = out["Province"].astype(str).str.strip().str.lower()
    out[target_col] = out[ur_col]
    out.loc[(out[ur_col] == "Urban") & province.isin(TEHRAN_LABELS), target_col] = "Urban_Tehran"
    return out


def infer_iran_year(file_name: str) -> int:
    stem = int(Path(file_name).stem)
    if stem >= 100:
        return 1000 + stem
    if stem >= 90:
        return 1300 + stem
    return 1400 + stem


In [ ]:
# Load the prepared analysis file.

master = clean_columns(pd.read_csv(DATA_DIR / "master.csv"))

required_columns = {
    "Year", "ID", "Urban_Rural", "Province", "Weight", "Tenure",
    "Income", "Gross_Expenditure", "housing_cost_adj",
}
missing_columns = sorted(required_columns.difference(master.columns))
if missing_columns:
    raise KeyError(f"master.csv is missing required columns: {missing_columns}")

print(f"master loaded: {master.shape[0]:,} rows x {master.shape[1]:,} columns")
print(f"years: {master['Year'].min()}-{master['Year'].max()}")


## 3. Housing Share by Geography

Housing cost is divided by gross expenditure for each household. The table reports weighted annual means for rural households, urban households, and urban Tehran.


In [ ]:
# Estimate housing shares by year and geography.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Weight", "Gross_Expenditure", "housing_cost_adj"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


df = df[(df["Weight"] > 0) & (df["Gross_Expenditure"] > 0)].copy()


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df["UR"] = pd.NA
df.loc[ur.str.startswith("u"), "UR"] = "Urban"
df.loc[ur.str.startswith("r"), "UR"] = "Rural"
df = df[df["UR"].isin(["Urban", "Rural"])].copy()


df["housing_share"] = (
    df["housing_cost_adj"] / df["Gross_Expenditure"]
)


prov = df["Province"].astype(str).str.strip().str.lower()

df["UR_EXT"] = df["UR"]
df.loc[
    (df["UR"] == "Urban") & (prov.isin(TEHRAN_LABELS)),
    "UR_EXT"
] = "Urban_Tehran"


tab_long = (
    df.groupby(["Year", "UR_EXT"], as_index=False)
      .apply(lambda x: pd.Series({
          "housing_share_wmean": np.average(x["housing_share"], weights=x["Weight"])
      }))
)


tab_wide = (
    tab_long.pivot(index="Year", columns="UR_EXT", values="housing_share_wmean")
            .reset_index()
            .rename_axis(None, axis=1)
            .sort_values("Year")
)

tab_housing_share = tab_wide


The ratio output is converted to percentages for presentation.


In [ ]:
# Express housing shares as percentages.

tab_percent = tab_wide.copy()


for c in ["Urban", "Rural"]:
    if c in tab_percent.columns:
        tab_percent[c] = tab_percent[c] * 100


tab_percent[["Urban", "Rural"]] = tab_percent[["Urban", "Rural"]].round(1)

tab_housing_share_percent = tab_percent


## 4. Renter-to-Owner Ratio

This table compares the weighted size of the renter population with the weighted size of the owner population by year and geography.


In [ ]:
# Compute renter-to-owner ratios using survey weights.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
df = df[df["Weight"] > 0].copy()


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df["UR"] = pd.NA
df.loc[ur.str.startswith("u"), "UR"] = "Urban"
df.loc[ur.str.startswith("r"), "UR"] = "Rural"
df = df[df["UR"].isin(["Urban", "Rural"])].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("rent|mortg"), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own"), "tenure_group"] = "Owner"
df = df[df["tenure_group"].isin(["Renter", "Owner"])].copy()


prov = df["Province"].astype(str).str.strip().str.lower()
df["UR_EXT"] = df["UR"]
df.loc[
    (df["UR"] == "Urban") & (prov.isin(TEHRAN_LABELS)),
    "UR_EXT"
] = "Urban_Tehran"


w = (
    df.groupby(["Year", "UR_EXT", "tenure_group"])["Weight"]
      .sum()
      .reset_index(name="w_sum")
)


pivot = (
    w.pivot(index=["Year", "UR_EXT"], columns="tenure_group", values="w_sum")
     .reset_index()
)


pivot["renter_to_owner"] = pivot["Renter"] / pivot["Owner"]


out = (
    pivot.pivot(index="Year", columns="UR_EXT", values="renter_to_owner")
         .reset_index()
         .rename_axis(None, axis=1)
         .rename(columns={
             "Urban": "Urban_Renter_to_Owner",
             "Rural": "Rural_Renter_to_Owner",
             "Urban_Tehran": "Tehran_Renter_to_Owner"
         })
         .sort_values("Year")
)

tab_renter_to_owner_ratio = out


## 5. Housing Cost Relative to Income

For urban households, adjusted housing cost is divided by income and averaged by tenure group. This gives a direct measure of housing pressure for renters and owners.


In [ ]:
# Estimate housing pressure for urban owners and renters.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in [
    "Weight",
    "Income",
    "housing_cost_adj"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


df = df[(df["Weight"] > 0) & (df["Income"] > 0)].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("rent|mortg"), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own"), "tenure_group"] = "Owner"
df = df[df["tenure_group"].isin(["Renter", "Owner"])].copy()


df["housing_income_ratio"] = (
    df["housing_cost_adj"] / df["Income"]
)


tab = (
    df.groupby(["Year", "tenure_group"])
      .apply(lambda x: np.average(
          x["housing_income_ratio"], weights=x["Weight"]
      ))
      .reset_index(name="ratio_wmean")
)


out = (
    tab.pivot(index="Year", columns="tenure_group", values="ratio_wmean")
       .reset_index()
       .rename_axis(None, axis=1)
       .rename(columns={
           "Renter": "Urban_Renter_%",
           "Owner": "Urban_Owner_%"
       })
       .sort_values("Year")
)

out["Urban_Renter_%"] *= 100
out["Urban_Owner_%"] *= 100


out[["Urban_Renter_%", "Urban_Owner_%"]] = out[
    ["Urban_Renter_%", "Urban_Owner_%"]
].round(1)

tab_urban_housing_pressure_by_tenure = out


## 6. Tehran Housing Share by Tenure

The same housing-share calculation is repeated for urban Tehran, separately for owners and renters.


In [ ]:
# Estimate Tehran housing shares by tenure.

import pandas as pd
import numpy as np


df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Weight", "Gross_Expenditure", "housing_cost_adj"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


df = df[(df["Weight"] > 0) & (df["Gross_Expenditure"] > 0)].copy()


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
prov = df["Province"].astype(str).str.strip().str.lower()

df = df[
    ur.str.startswith("u") &
    prov.isin(TEHRAN_LABELS)
].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = np.nan
df.loc[ten.str.contains("own", na=False), "tenure_group"] = "Owner"
df.loc[ten.str.contains("rent|mortg", na=False), "tenure_group"] = "Renter"
df = df[df["tenure_group"].isin(["Owner", "Renter"])].copy()


df["housing_share"] = df["housing_cost_adj"] / df["Gross_Expenditure"]


tab = (
    df.groupby(["Year", "tenure_group"])
      .apply(lambda g: np.average(g["housing_share"], weights=g["Weight"]))
      .reset_index(name="housing_share_wmean")
)


out = (
    tab.pivot(index="Year", columns="tenure_group", values="housing_share_wmean")
       .reset_index()
       .rename_axis(None, axis=1)
       .rename(columns={
           "Owner": "Tehran_Owner_%",
           "Renter": "Tehran_Renter_%"
       })
       .sort_values("Year")
)


out[["Tehran_Owner_%", "Tehran_Renter_%"]] = (
    out[["Tehran_Owner_%", "Tehran_Renter_%"]] * 100
).round(1)

tab_tehran_housing_share_by_tenure = out


## 7. Rental Income Among Owners

This table measures rent received by urban owner households as a share of total household income.


In [ ]:
# Measure rent income as a share of owner-household income.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Weight", "Income", "Rent"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df = df[ten.str.contains("own")].copy()


df = df[(df["Weight"] > 0) & (df["Income"] > 0)].copy()


df["rent_income_ratio"] = df["Rent"] / df["Income"]


out = (
    df.groupby("Year")
      .apply(lambda x: np.average(
          x["rent_income_ratio"], weights=x["Weight"]
      ))
      .reset_index(name="Urban_Owner_Rent_to_Income_%")
      .sort_values("Year")
)


out["Urban_Owner_Rent_to_Income_%"] *= 100
out["Urban_Owner_Rent_to_Income_%"] = out[
    "Urban_Owner_Rent_to_Income_%"
].round(2)

tab_urban_owner_rent_income_share = out


## 8. Real Income and Housing Cost for Urban Renters

CPI is merged to urban renter observations so that income and adjusted housing costs can be expressed in real terms before annual weighted means are calculated.


In [ ]:
# Convert renter income and housing costs to real terms.

import pandas as pd
import numpy as np


cpi = pd.read_excel(DATA_DIR / "CPI.xlsx")
cpi.columns = cpi.columns.astype(str).str.strip()

cpi = cpi.rename(columns={

    "سال": "Year",
    "CPI": "CPI",
    "شاخص قیمت مصرف کننده (CPI)": "CPI"
})

cpi["CPI"] = pd.to_numeric(cpi["CPI"], errors="coerce")


df = master.copy()
df.columns = df.columns.astype(str).str.strip()

for c in [
    "Weight",
    "Income",
    "housing_cost_adj"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df = df[ten.str.contains("rent|mortg")].copy()


df = df[
    (df["Weight"] > 0) &
    (df["Income"] > 0)
].copy()


df = df.merge(cpi[["Year", "CPI"]], on="Year", how="left")
df = df[df["CPI"].notna()].copy()


df["real_income"] = df["Income"] / df["CPI"]
df["real_housing_cost"] = (
    df["housing_cost_adj"] / df["CPI"]
)


annual = (
    df.groupby("Year")
      .apply(lambda x: pd.Series({
          "real_income_wmean": np.average(
              x["real_income"], weights=x["Weight"]
          ),
          "real_housing_cost_wmean": np.average(
              x["real_housing_cost"], weights=x["Weight"]
          )
      }))
      .reset_index()
      .sort_values("Year")
)


annual["ΔReal_Income_%"] = (
    annual["real_income_wmean"].pct_change() * 100
)

annual["ΔReal_Housing_Cost_%"] = (
    annual["real_housing_cost_wmean"].pct_change() * 100
)

out = annual[[
    "Year",
    "ΔReal_Income_%",
    "ΔReal_Housing_Cost_%"
]].round(2)

tab_real_urban_renter_income_housing = out


## 9. Real Annual Summary Tables

This section creates four real-value summary tables: urban renters, urban owners, Tehran renters, and Tehran owners. Income and gross expenditure are scaled to 10 million rials, while adjusted housing cost is scaled to 1 million rials.


In [ ]:
# Create real weighted summary tables.

import pandas as pd
import numpy as np


cpi = pd.DataFrame([
    (1389,100.0000),(1390,126.2934),(1391,160.7169),(1392,219.5442),
    (1393,256.0029),(1394,287.9641),(1395,308.8283),(1396,333.6733),
    (1397,393.7816),(1398,550.9294),(1399,719.4815),(1400,1031.6580),
    (1401,1480.3100),(1402,2140.219429),(1403,2834.846295)
], columns=["Year","CPI"])


df = master.copy()
df.columns = df.columns.astype(str).str.strip()

for c in ["Year","Weight","Income","Gross_Expenditure","housing_cost_adj"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[(df["Weight"] > 0) & df["Year"].notna()].copy()


ur = df["Urban_Rural"].astype(str).str.lower().str.strip()
df = df[ur.str.startswith("u")].copy()

ten = df["Tenure"].astype(str).str.lower().str.strip()
df["tenure_group"] = np.nan
df.loc[ten.str.contains("rent|mortg", na=False), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own", na=False), "tenure_group"] = "Owner"
df = df[df["tenure_group"].isin(["Owner","Renter"])].copy()

prov = df["Province"].astype(str).str.lower().str.strip()
df["is_tehran"] = prov.isin(TEHRAN_LABELS)


df = df.merge(cpi, on="Year", how="left")
df = df[df["CPI"].notna()].copy()

df["Income_real"]            = df["Income"]            * (100 / df["CPI"])
df["Gross_Expenditure_real"] = df["Gross_Expenditure"] * (100 / df["CPI"])
df["housing_cost_adj_real"]  = df["housing_cost_adj"]  * (100 / df["CPI"])


def wmean(x, w):
    m = x.notna() & w.notna() & (w > 0)
    return np.average(x[m], weights=w[m]) if m.any() else np.nan

def build_table(dfg):
    out = (
        dfg.groupby("Year")
           .apply(lambda g: pd.Series({

               "Income_real_wmean":            wmean(g["Income_real"], g["Weight"]) / 10_000_000,
               "Gross_Expenditure_real_wmean": wmean(g["Gross_Expenditure_real"], g["Weight"]) / 10_000_000,
               "housing_cost_adj_real_wmean":  wmean(g["housing_cost_adj_real"], g["Weight"]) / 10_000_000,
           }))
           .reset_index()
           .sort_values("Year")
    )
    return out


df_tehran_renter = build_table(df[(df["is_tehran"]) & (df["tenure_group"]=="Renter")])
df_tehran_owner  = build_table(df[(df["is_tehran"]) & (df["tenure_group"]=="Owner")])
df_urban_renter  = build_table(df[df["tenure_group"]=="Renter"])
df_urban_owner   = build_table(df[df["tenure_group"]=="Owner"])


df_tehran_renter


## 10. Tenure Composition by Income Decile

Within each modified OECD income decile, the table reports the weighted share of urban households that are owners and renters.


In [ ]:
# Estimate tenure composition across income deciles.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
df = df[df["Weight"] > 0].copy()


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


df["Income_Decile_OECD_Modified"] = pd.to_numeric(
    df["Income_Decile_OECD_Modified"], errors="coerce"
)
df = df[df["Income_Decile_OECD_Modified"].between(1, 10)].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("rent|mortg"), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own"), "tenure_group"] = "Owner"

df = df[df["tenure_group"].isin(["Owner", "Renter"])].copy()


w = (
    df.groupby(["Income_Decile_OECD_Modified", "tenure_group"])["Weight"]
      .sum()
      .reset_index(name="w_sum")
)


total = (
    w.groupby("Income_Decile_OECD_Modified")["w_sum"]
     .sum()
     .reset_index(name="w_total")
)

w = w.merge(total, on="Income_Decile_OECD_Modified", how="left")
w["share_pct"] = 100 * w["w_sum"] / w["w_total"]


out = (
    w.pivot(
        index="Income_Decile_OECD_Modified",
        columns="tenure_group",
        values="share_pct"
    )
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns={
        "Income_Decile_OECD_Modified": "Decile",
        "Owner": "Owner_%",
        "Renter": "Renter_%"
    })
    .sort_values("Decile")
)


out[["Owner_%", "Renter_%"]] = out[["Owner_%", "Renter_%"]].round(1)

tab_urban_tenure_by_income_decile = out


## 11. Expenditure by Category

This table reports annual weighted mean expenditure for selected spending categories.


In [ ]:
# Compute annual weighted mean expenditure by category.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


COLS = {
    "Clothing and Footwear": "Clothing and Footwear",
    "Housing, Water, Electricity, Gas and Other Fuels": "housing_cost_adj",
    "Furnishings and Household Maintenance": "Furnishings Household Equipment and Routine Household Maintenance",
    "Health": "Health",
    "Transport": "Transport",
    "Information and Communication": "Information and Communication",
    "Recreation and Culture": "Recreation and Culture",
    "Education Services": "Education Services",
    "Food and Non-Alcoholic Beverages": "Food and Non-Alcoholic Beverages",
}


df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
for fa, en in COLS.items():
    df[en] = pd.to_numeric(df[en], errors="coerce")


df = df[df["Weight"] > 0].copy()

def wmean(s, w):
    m = s.notna() & w.notna() & (w > 0)
    if m.sum() == 0:
        return np.nan
    return np.average(s[m], weights=w[m])


out = df.groupby("Year").apply(
    lambda g: pd.Series({fa: wmean(g[en], g["Weight"]) for fa, en in COLS.items()})
).reset_index().sort_values("Year")

tab_expenditure_by_category = out


## 12. Income Deciles Before and After Housing Costs

This section compares household rank in the income distribution before and after subtracting housing costs. Deciles are assigned within each year using survey weights.


In [ ]:
# Compare income deciles before and after housing costs.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Year", "Weight", "Income", "housing_cost_adj"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[(df["Weight"] > 0) & df["Year"].notna()].copy()


df["housing_cost"] = pd.to_numeric(df["housing_cost_adj"], errors="coerce")
df["income_after_housing"] = df["Income"] - df["housing_cost"]


df.loc[df["income_after_housing"] < 0, "income_after_housing"] = 0


def assign_weighted_deciles(values: pd.Series, weights: pd.Series) -> pd.Series:
    v = values.to_numpy()
    w = weights.to_numpy()

    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    out = np.full(len(v), np.nan)

    if ok.sum() == 0:
        return pd.Series(out, index=values.index)

    idx = np.where(ok)[0]
    order = idx[np.argsort(v[ok], kind="mergesort")]

    w_sorted = w[order]
    cum = np.cumsum(w_sorted) / w_sorted.sum()


    d_sorted = np.ceil(10 * cum).astype(int)
    d_sorted[d_sorted < 1] = 1
    d_sorted[d_sorted > 10] = 10

    out[order] = d_sorted
    return pd.Series(out, index=values.index)


df["decile_before"] = df.groupby("Year", group_keys=False).apply(
    lambda g: assign_weighted_deciles(g["Income"].fillna(0), g["Weight"])
)
df["decile_after"] = df.groupby("Year", group_keys=False).apply(
    lambda g: assign_weighted_deciles(g["income_after_housing"].fillna(0), g["Weight"])
)


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
dfu = df[ur.str.startswith("u")].copy()

ten = dfu["Tenure"].astype(str).str.strip().str.lower()
dfu["tenure_group"] = pd.NA
dfu.loc[ten.str.contains("rent|mortg"), "tenure_group"] = "Renter"
dfu.loc[ten.str.contains("own"), "tenure_group"] = "Owner"
dfu = dfu[dfu["tenure_group"].isin(["Owner", "Renter"])].copy()


dfu["decile_drop_ratio"] = (
    (dfu["decile_before"] - dfu["decile_after"]).clip(lower=0) / dfu["decile_before"]
)


tab = (
    dfu.dropna(subset=["decile_drop_ratio", "decile_before", "decile_after"])
       .groupby(["Year", "tenure_group"])
       .apply(lambda g: np.average(g["decile_drop_ratio"], weights=g["Weight"]) * 100)
       .reset_index(name="Decile_Drop_%")
)


out = (
    tab.pivot(index="Year", columns="tenure_group", values="Decile_Drop_%")
       .reset_index()
       .rename_axis(None, axis=1)
       .rename(columns={"Owner": "Owner_DecileDrop_%", "Renter": "Renter_DecileDrop_%"})
       .sort_values("Year")
)

out[["Owner_DecileDrop_%", "Renter_DecileDrop_%"]] = out[["Owner_DecileDrop_%", "Renter_DecileDrop_%"]].round(1)

tab_income_decile_shift_after_housing = out


## 13. Housing Poverty Thresholds

Urban renters are classified as housing poor when adjusted housing costs exceed 30 percent or 40 percent of income. The output reports weighted annual rates at both thresholds.


In [ ]:
# Estimate housing-poverty rates for urban renters.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Weight", "Income", "housing_cost_adj"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df = df[ten.str.contains("rent|mortg")].copy()


df = df[(df["Weight"] > 0) & (df["Income"] > 0)].copy()


df["housing_pressure"] = df["housing_cost_adj"] / df["Income"]


df["poor_30"] = (df["housing_pressure"] >= 0.30).astype(int)
df["poor_40"] = (df["housing_pressure"] >= 0.40).astype(int)


out = (
    df.groupby("Year")
      .apply(lambda g: pd.Series({
          "Housing_Poverty_30_%": 100 * np.average(g["poor_30"], weights=g["Weight"]),
          "Housing_Poverty_40_%": 100 * np.average(g["poor_40"], weights=g["Weight"]),
      }))
      .reset_index()
      .sort_values("Year")
)

out[["Housing_Poverty_30_%", "Housing_Poverty_40_%"]] = out[
    ["Housing_Poverty_30_%", "Housing_Poverty_40_%"]
].round(1)

tab_housing_poverty_urban_renters = out


## 14. Non-Cash Employment Income

The yearly files are read again to extract `NonCash_Employment_Income`, which is then used in the following section.


In [ ]:
# Extract non-cash employment income from yearly files.

import pandas as pd
from pathlib import Path


FILES = [
    "94.csv","95.csv","96.csv","97.csv","98.csv","99.csv",
    "400.csv","401.csv","402.csv","403.csv",
]
FILES = [f for f in FILES if Path(f).exists()]

if not FILES:
    raise FileNotFoundError("No yearly CSV files were found next to the notebook.")


TARGET = "NonCash_Employment_Income"
ALIASES = [
    "NonCash_Employment_Income",
    "NonCash Employment Income",
    "NonCash_Employment Income",
    "Noncash_Employment_Income",
    "NonCash_EmploymentIncome",
]


KEYS = ["Year", "ID", "Weight", "Urban_Rural", "Tenure"]


rows = []
missing_files = []

for fname in FILES:
    df = pd.read_csv(DATA_DIR / fname)
    df.columns = df.columns.astype(str).str.strip()


    if "Year" not in df.columns:
        yy = int(Path(fname).stem)
        if yy >= 100:
            year = 1000 + yy
        elif yy >= 90:
            year = 1300 + yy
        else:
            year = 1400 + yy
        df["Year"] = year


    col = next((c for c in ALIASES if c in df.columns), None)
    if col is None:
        missing_files.append(fname)
        continue


    keep = [c for c in KEYS if c in df.columns] + [col]
    tmp = df[keep].copy()


    if col != TARGET:
        tmp = tmp.rename(columns={col: TARGET})

    tmp["__source_file__"] = fname
    rows.append(tmp)

if not rows:
    raise ValueError("NonCash_Employment_Income was not found in any yearly file.")

noncash_emp = pd.concat(rows, ignore_index=True)

print("✅ extracted dataframe: noncash_emp")
print("rows:", len(noncash_emp), "| cols:", noncash_emp.shape[1])

if missing_files:
    print("\nFiles without the target column:")
    for f in missing_files:
        print(" -", f)

noncash_emp.head()


## 15. Non-Cash Employment Income Among Housing-Poor Renters

This table focuses on urban renters and summarizes the employment-income proxy among households above the housing-poverty thresholds.


In [ ]:
# Summarize non-cash employment income among housing-poor renters.

import numpy as np
import pandas as pd


df = noncash_emp.copy()
df.columns = df.columns.astype(str).str.strip()


for c in ["Weight", "NonCash_Employment_Income"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()

ten = df["Tenure"].astype(str).str.strip().str.lower()
df = df[ten.str.contains("rent|mortg")].copy()


df = df[df["Weight"] > 0].copy()


base = master[[
    "Year", "ID",
    "Income",
    "housing_cost_adj"
]].copy()

base.columns = base.columns.astype(str).str.strip()

for c in ["Income", "housing_cost_adj"]:
    base[c] = pd.to_numeric(base[c], errors="coerce")


df = df.merge(
    base,
    on=["Year", "ID"],
    how="left"
)

df = df[(df["Income"] > 0)].copy()

df["housing_pressure"] = (
    df["housing_cost_adj"] / df["Income"]
)


df["poor_30"] = df["housing_pressure"] >= 0.30
df["poor_40"] = df["housing_pressure"] >= 0.40


df["employed"] = df["NonCash_Employment_Income"] > 0


out = (
    df.groupby("Year")
      .apply(lambda g: pd.Series({
          "Employed_Among_Poor_30_%": (
              100 * np.average(
                  g.loc[g["poor_30"], "employed"],
                  weights=g.loc[g["poor_30"], "Weight"]
              ) if g["poor_30"].any() else np.nan
          ),
          "Employed_Among_Poor_40_%": (
              100 * np.average(
                  g.loc[g["poor_40"], "employed"],
                  weights=g.loc[g["poor_40"], "Weight"]
              ) if g["poor_40"].any() else np.nan
          ),
      }))
      .reset_index()
      .sort_values("Year")
)

out[["Employed_Among_Poor_30_%", "Employed_Among_Poor_40_%"]] = out[
    ["Employed_Among_Poor_30_%", "Employed_Among_Poor_40_%"]
].round(1)

tab_noncash_employment_housing_poor_renters = out


## 16. Disposable Income After Housing

Disposable income after housing is defined as income minus adjusted housing cost. Negative values are set to zero before weighted annual averages are estimated by tenure.


In [ ]:
# Estimate disposable income after housing by tenure.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in [
    "Weight",
    "Income",
    "housing_cost_adj"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("own"), "tenure_group"] = "Owner"
df.loc[ten.str.contains("rent|mortg"), "tenure_group"] = "Renter"
df = df[df["tenure_group"].isin(["Owner", "Renter"])].copy()


df["disposable_income"] = (
    df["Income"] - df["housing_cost_adj"]
)


df.loc[df["disposable_income"] < 0, "disposable_income"] = 0


df = df[
    (df["Weight"] > 0) &
    (df["Income"] > 0)
].copy()


tab = (
    df.groupby(["Year", "tenure_group"])
      .apply(lambda g: np.average(
          g["disposable_income"], weights=g["Weight"]
      ))
      .reset_index(name="Disposable_Income_WMean")
)


out = (
    tab.pivot(index="Year", columns="tenure_group", values="Disposable_Income_WMean")
       .reset_index()
       .rename_axis(None, axis=1)
       .rename(columns={
           "Owner": "Owner_Disposable_Income",
           "Renter": "Renter_Disposable_Income"
       })
       .sort_values("Year")
)


for c in ["Owner_Disposable_Income", "Renter_Disposable_Income"]:
    out[c] = out[c] / 10_000_000


out[["Owner_Disposable_Income", "Renter_Disposable_Income"]] = (
    out[["Owner_Disposable_Income", "Renter_Disposable_Income"]]
    .round(1)
)

tab_disposable_income_after_housing = out


## 17. Ownership and Renting at the Bottom and Top of the Distribution

This section compares owner and renter shares in selected income-decile bands, especially the first and tenth deciles and broader bottom/top groups.


In [ ]:
# Compare tenure shares in lower and upper decile groups.

import pandas as pd
import numpy as np

df = master.copy()
df.columns = df.columns.astype(str).str.strip()


df["Weight"] = pd.to_numeric(df["Weight"], errors="coerce")
df["Income_Decile_OECD_Modified"] = pd.to_numeric(df["Income_Decile_OECD_Modified"], errors="coerce")


df = df[(df["Weight"] > 0)].copy()

ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()

df = df[df["Income_Decile_OECD_Modified"].between(1, 10)].copy()


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("rent|mortg", na=False), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own", na=False), "tenure_group"] = "Owner"
df = df[df["tenure_group"].isin(["Owner", "Renter"])].copy()


def shares_in_group(g: pd.DataFrame) -> pd.Series:
    w_owner = g.loc[g["tenure_group"] == "Owner", "Weight"].sum()
    w_renter = g.loc[g["tenure_group"] == "Renter", "Weight"].sum()
    w_total = w_owner + w_renter
    if w_total == 0:
        return pd.Series({"Owner_%": np.nan, "Renter_%": np.nan})
    return pd.Series({
        "Owner_%": 100 * w_owner / w_total,
        "Renter_%": 100 * w_renter / w_total
    })


df_tb = df.copy()
df_tb["decile_band"] = pd.NA
df_tb.loc[df_tb["Income_Decile_OECD_Modified"].between(1, 3), "decile_band"] = "Bottom_1_3"
df_tb.loc[df_tb["Income_Decile_OECD_Modified"].between(8, 10), "decile_band"] = "Top_8_10"
df_tb = df_tb[df_tb["decile_band"].isin(["Bottom_1_3", "Top_8_10"])].copy()

tab_tb = (
    df_tb.groupby(["Year", "decile_band"])
         .apply(shares_in_group)
         .reset_index()
)

df_share_topbottom = (
    tab_tb.pivot(index="Year", columns="decile_band", values=["Owner_%", "Renter_%"])
          .reset_index()
)


df_share_topbottom.columns = [
    "Year" if c[0] == "Year" else f"{c[1]}_{c[0]}"
    for c in df_share_topbottom.columns
]


df_share_topbottom = df_share_topbottom[[
    "Year",
    "Bottom_1_3_Owner_%", "Bottom_1_3_Renter_%",
    "Top_8_10_Owner_%", "Top_8_10_Renter_%"
]].sort_values("Year")

for c in ["Bottom_1_3_Owner_%", "Bottom_1_3_Renter_%", "Top_8_10_Owner_%", "Top_8_10_Renter_%"]:
    df_share_topbottom[c] = df_share_topbottom[c].round(1)


df_110 = df[df["Income_Decile_OECD_Modified"].isin([1, 10])].copy()
df_110["decile_band"] = df_110["Income_Decile_OECD_Modified"].map({1: "Decile_1", 10: "Decile_10"})

tab_110 = (
    df_110.groupby(["Year", "decile_band"])
          .apply(shares_in_group)
          .reset_index()
)

df_share_decile1_10 = (
    tab_110.pivot(index="Year", columns="decile_band", values=["Owner_%", "Renter_%"])
           .reset_index()
)

df_share_decile1_10.columns = [
    "Year" if c[0] == "Year" else f"{c[1]}_{c[0]}"
    for c in df_share_decile1_10.columns
]

df_share_decile1_10 = df_share_decile1_10[[
    "Year",
    "Decile_1_Owner_%", "Decile_1_Renter_%",
    "Decile_10_Owner_%", "Decile_10_Renter_%"
]].sort_values("Year")

for c in ["Decile_1_Owner_%", "Decile_1_Renter_%", "Decile_10_Owner_%", "Decile_10_Renter_%"]:
    df_share_decile1_10[c] = df_share_decile1_10[c].round(1)


df_share_topbottom,


## 18. Expenditure-Inflation Exposure

Household budget shares for food, housing, transport, and health are combined with annual category-specific inflation rates. The resulting measure approximates exposure to inflation through each household's expenditure structure, with separate outputs for Tehran and other urban areas.


In [ ]:
# Combine budget shares with category inflation rates.

import pandas as pd
import numpy as np


inflation = pd.DataFrame({
    "Year": [1394,1395,1396,1397,1398,1399,1400,1401,1402,1403],
    "Food_inf":      [9.7, 7.6,12.3,38.4,42.7,38.7,52.1,70.2,41.4,28.0],
    "Housing_inf":   [14.5,6.0,7.2,17.4,23.9,25.8,26.8,32.4,39.1,39.7],
    "Transport_inf": [10.0,5.3,4.9,28.1,47.2,66.8,37.7,35.7,42.1,27.8],
    "Health_inf":    [15.0,9.1,7.2,17.3,25.9,29.7,38.6,43.5,43.3,29.2],
})


df = master.copy()
df.columns = df.columns.astype(str).str.strip()


for c in [
    "Year","Weight","Gross_Expenditure",
    "Food and Non-Alcoholic Beverages",
    "housing_cost_adj",
    "Transport","Health"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[
    (df["Weight"] > 0) &
    (df["Gross_Expenditure"] > 0) &
    (df["Year"].between(1394,1403))
].copy()


ur = df["Urban_Rural"].astype(str).str.strip().str.lower()
df = df[ur.str.startswith("u")].copy()

prov = df["Province"].astype(str).str.strip().str.lower()
df["city_group"] = np.where(
    prov.isin(TEHRAN_LABELS),
    "Tehran",
    "Other_Cities"
)


ten = df["Tenure"].astype(str).str.strip().str.lower()
df["tenure_group"] = pd.NA
df.loc[ten.str.contains("rent|mortg", na=False), "tenure_group"] = "Renter"
df.loc[ten.str.contains("own", na=False), "tenure_group"] = "Owner"
df = df[df["tenure_group"].isin(["Owner","Renter"])].copy()


df = df.merge(inflation, on="Year", how="left")
df = df[df["Food_inf"].notna()].copy()


df["w_food"]      = df["Food and Non-Alcoholic Beverages"] / df["Gross_Expenditure"]
df["w_housing"]   = df["housing_cost_adj"] / df["Gross_Expenditure"]
df["w_transport"] = df["Transport"] / df["Gross_Expenditure"]
df["w_health"]    = df["Health"] / df["Gross_Expenditure"]

for c in ["w_food","w_housing","w_transport","w_health"]:
    df[c] = df[c].fillna(0)


df["experienced_inflation"] = (
    df["w_food"]      * df["Food_inf"] +
    df["w_housing"]   * df["Housing_inf"] +
    df["w_transport"] * df["Transport_inf"] +
    df["w_health"]    * df["Health_inf"]
)


tab = (
    df.groupby(["Year","city_group","tenure_group"])
      .apply(lambda g: np.average(
          g["experienced_inflation"], weights=g["Weight"]
      ))
      .reset_index(name="Experienced_Inflation_%")
      .round(2)
)


df_expinf_tehran = (
    tab[tab["city_group"]=="Tehran"]
    .pivot(index="Year", columns="tenure_group", values="Experienced_Inflation_%")
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns={
        "Owner":"Owner_Inflation_%",
        "Renter":"Renter_Inflation_%"
    })
    .sort_values("Year")
)


df_expinf_other_cities = (
    tab[tab["city_group"]=="Other_Cities"]
    .pivot(index="Year", columns="tenure_group", values="Experienced_Inflation_%")
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns={
        "Owner":"Owner_Inflation_%",
        "Renter":"Renter_Inflation_%"
    })
    .sort_values("Year")
)


df_expinf_tehran


## 19. Export Tables

The notebook writes final analytical tables to `outputs/tables/`. Keeping tables as separate files makes the repository easier to review and avoids storing bulky rendered outputs inside the notebook.


In [ ]:
# Export analytical tables as CSV files.

OUTPUT_DIR_CANDIDATES = [
    Path("../outputs/tables"),
    Path("outputs/tables"),
]
OUTPUT_DIR = next(
    (candidate.resolve() for candidate in OUTPUT_DIR_CANDIDATES if candidate.parent.exists()),
    OUTPUT_DIR_CANDIDATES[0].resolve(),
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "housing_share_by_geography.csv": tab_housing_share,
    "housing_share_by_geography_percent.csv": tab_housing_share_percent,
    "renter_to_owner_ratio.csv": tab_renter_to_owner_ratio,
    "urban_housing_pressure_by_tenure.csv": tab_urban_housing_pressure_by_tenure,
    "tehran_housing_share_by_tenure.csv": tab_tehran_housing_share_by_tenure,
    "urban_owner_rent_income_share.csv": tab_urban_owner_rent_income_share,
    "real_urban_renter_income_housing.csv": tab_real_urban_renter_income_housing,
    "real_urban_renter_summary.csv": df_urban_renter,
    "real_urban_owner_summary.csv": df_urban_owner,
    "real_tehran_renter_summary.csv": df_tehran_renter,
    "real_tehran_owner_summary.csv": df_tehran_owner,
    "urban_tenure_by_income_decile.csv": tab_urban_tenure_by_income_decile,
    "expenditure_by_category.csv": tab_expenditure_by_category,
    "income_decile_shift_after_housing.csv": tab_income_decile_shift_after_housing,
    "housing_poverty_urban_renters.csv": tab_housing_poverty_urban_renters,
    "noncash_employment_housing_poor_renters.csv": tab_noncash_employment_housing_poor_renters,
    "disposable_income_after_housing.csv": tab_disposable_income_after_housing,
    "tenure_share_deciles_1_10.csv": df_share_decile1_10,
    "tenure_share_top_bottom_deciles.csv": df_share_topbottom,
    "inflation_exposure_tehran.csv": df_expinf_tehran,
    "inflation_exposure_other_cities.csv": df_expinf_other_cities,
}

for file_name, table in tables.items():
    table.to_csv(OUTPUT_DIR / file_name, index=False)

export_index = pd.DataFrame(
    {
        "file": list(tables.keys()),
        "rows": [len(table) for table in tables.values()],
        "columns": [len(table.columns) for table in tables.values()],
    }
)
export_index.to_csv(OUTPUT_DIR / "table_index.csv", index=False)

print(f"Exported {len(tables)} tables to {OUTPUT_DIR}")
